## Notebook 2: Tokenization, Vocabulary, and Vector Space Model (F2 → F4)

In [ ]:
import pandas as pd
import os
import re
from sklearn.feature_extraction.text import TfidfTransformer
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("stopwords")
nltk.download("wordnet")
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

In [2]:
LIBRARY = pd.read_csv("../data/OutputTables/LIBRARY.csv", index_col="book_id")
LIBRARY.head()

,title,author,subjects,bookshelves,genre,char_count,word_count,source_url
book_id,,,,,,,,
62,A PRINCESS OF MARS,"BURROUGHS, EDGAR RICE",SCIENCE FICTION\nMARS (PLANET) -- FICTION\nDEJ...,SCIENCE FICTION\nBEST BOOKS EVER LISTINGS,science fiction,378163,67436,https://www.gutenberg.org/ebooks/62
64,THE GODS OF MARS,"BURROUGHS, EDGAR RICE",PS\nLIFE ON OTHER PLANETS -- FICTION\nDEJAH TH...,SCIENCE FICTION,science fiction,459695,82741,https://www.gutenberg.org/ebooks/64
68,WARLORD OF MARS,"BURROUGHS, EDGAR RICE","CARTER, JOHN (FICTITIOUS CHARACTER) -- FICTION...",SCIENCE FICTION,science fiction,317806,57134,https://www.gutenberg.org/ebooks/68
72,"THUVIA, MAID OF MARS","BURROUGHS, EDGAR RICE",SCIENCE FICTION\nPS\nMARS (PLANET) -- FICTION\...,SCIENCE FICTION,science fiction,272243,47143,https://www.gutenberg.org/ebooks/72
83,"FROM THE EARTH TO THE MOON; AND, ROUND THE MOON","VERNE, JULES",MOON -- FICTION\nPQ\nMANNED SPACE FLIGHT -- FI...,SCIENCE FICTION\nMOVIE BOOKS,science fiction,556261,91500,https://www.gutenberg.org/ebooks/83


In [3]:
texts = {}
for book_id in LIBRARY.index:
    path = f"../data/CleanTexts/{book_id}.txt"  # <-- changed
    with open(path, "r", encoding="utf-8") as f:
        texts[book_id] = f.read()

In [4]:
OHCO = ['book_id', 'chapter_num', 'para_num', 'sent_num']

rows = []

for book_id, book in LIBRARY.iterrows():
    text = texts[book_id]
    
    chapters = re.split(r'\n\s*(CHAPTER|Chapter)\s+[IVXLC\d]+', text)
    
    for chap_num, chapter_text in enumerate(chapters):
        paragraphs = re.split(r'\n\s*\n', chapter_text.strip())
        
        for para_num, para_text in enumerate(paragraphs):
            sentences = sent_tokenize(para_text)
            
            for sent_num, sentence in enumerate(sentences):
                tokens = word_tokenize(sentence)
                tagged = pos_tag(tokens)
                
                for tok_num, (token, pos) in enumerate(tagged):
                    token_lower = token.lower()
                    rows.append({
                        "book_id": book_id,
                        "chapter_num": chap_num,
                        "para_num": para_num,
                        "sent_num": sent_num,
                        "tok_num": tok_num,
                        "token": token,
                        "term_str": token_lower,
                        "lemma": lemmatizer.lemmatize(token_lower),
                        "pos": pos,
                        "is_stop": token_lower in stop_words,
                        "is_alpha": token.isalpha()
                    })

TOKEN = pd.DataFrame(rows)
TOKEN = TOKEN.set_index(OHCO)

In [5]:
TOKEN.to_csv("../data/OutputTables/TOKEN.csv")
print(TOKEN.shape)


(2374432, 7)


In [6]:
TOKEN_WORDS = TOKEN[
    (TOKEN["is_alpha"] == True) &
    (TOKEN["is_stop"] == False)
].copy()

TOKEN_WORDS.head()

tok_num         token      term_str  \
book_id chapter_num para_num sent_num                                        
62      0           0        0               1  Illustration  illustration   
                    1        0               1      Princess      princess   
                             0               3          Mars          mars   
                    2        0               1         Edgar         edgar   
                             0               2          Rice          rice   

                                              lemma  pos  is_stop  is_alpha  
book_id chapter_num para_num sent_num                                        
62      0           0        0         illustration  NNP    False      True  
                    1        0             princess  NNP    False      True  
                             0                  mar  NNP    False      True  
                    2        0                edgar  NNP    False      True  
                             0                 rice  NNP    False      True

In [ ]:
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

sent_sentiments = []
for book_id, book in LIBRARY.iterrows():
    for sent_num, sentence in enumerate(sent_tokenize(texts[book_id])):
        scores = sia.polarity_scores(sentence)
        sent_sentiments.append({"book_id": book_id, "sent_num": sent_num, **scores})

SENT_SENTIMENT = pd.DataFrame(sent_sentiments)

doc_sentiment = SENT_SENTIMENT.groupby("book_id")[["neg","neu","pos","compound"]].mean()
doc_sentiment.columns = ["sent_neg", "sent_neu", "sent_pos", "sent_compound"]

LIBRARY = LIBRARY.join(doc_sentiment)

SENT_SENTIMENT.to_csv("../data/OutputTables/SENT_SENTIMENT.csv")
LIBRARY.to_csv("../data/OutputTables/LIBRARY.csv") 

In [8]:
VOCAB = (
    TOKEN_WORDS.reset_index()
    .groupby("lemma")
    .agg(
        term_freq=("lemma", "count"),
        doc_freq=("book_id", "nunique")
    )
    .sort_values("term_freq", ascending=False)
)

VOCAB.index.name = "term_str"
VOCAB.head(20)

,term_freq,doc_freq
term_str,,
would,7108,51
one,6845,51
could,5791,51
upon,5604,48
said,5477,50
time,4053,51
u,3860,51
two,3245,51
man,3125,51


In [9]:
BOW = (
    TOKEN_WORDS.reset_index()
    .groupby(["book_id", "lemma"])
    .size()
    .unstack(fill_value=0)
)

In [10]:
BOW.to_csv("../data/OutputTables/BOW.csv")
print(BOW.shape)

(51, 30885)


In [11]:
tfidf_model = TfidfTransformer()
TFIDF_matrix = tfidf_model.fit_transform(BOW)

TFIDF = pd.DataFrame(
    TFIDF_matrix.toarray(),
    index=BOW.index,
    columns=BOW.columns
)

TFIDF.head()

lemma,aaaah,aaagh,aaah,aaanthor,aah,ab,aback,abaddon,abandon,abandoned,...,zulma,zulthran,zurb,zydanowycz,à,ægri,æolian,æolus,œdipus,δ
book_id,,,,,,,,,,,,,,,,,,,,,
62,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
64,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.001837,0.000811,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
68,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.001180,0.002086,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
72,0.0,0.0,0.0,0.065744,0.0,0.0,0.0,0.0,0.000000,0.000856,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
83,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.002008,0.001331,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001065


In [12]:
tfidf_mean_per_term = TFIDF.mean(axis=0)
VOCAB['tfidf_mean'] = VOCAB.index.map(tfidf_mean_per_term)

In [13]:
TFIDF.to_csv("../data/OutputTables/TFIDF.csv")
VOCAB.to_csv("../data/OutputTables/VOCAB.csv")